# Sentiment Analyzer — Reviewed Results Walkthrough

This companion notebook documents the verified outputs already present in the repository without replacing the original exploration, training, or evaluation notebooks.

Documentation standard used here:
- explain why a step matters before the code;
- keep code comments focused on analytical intent;
- show the recorded output;
- follow the output immediately with a plain-language interpretation;
- separate measured results from limitations and assumptions.


## 1. Confirm the dataset scale and class balance

The original exploration notebook uses the IMDb dataset with 25,000 labeled training reviews and 25,000 labeled test reviews. The training labels are evenly split between negative and positive reviews.


In [1]:
# Keep the dataset facts explicit so a reviewer can see the modeling context.
dataset_summary = {
    "train_reviews": 25_000,
    "test_reviews": 25_000,
    "negative_train": 12_500,
    "positive_train": 12_500,
}
for key, value in dataset_summary.items():
    print(f"{key}: {value:,}")


train_reviews: 25,000
test_reviews: 25,000
negative_train: 12,500
positive_train: 12,500


### What this tells us

The training split is balanced, so the headline accuracy is not being inflated by a dominant class. Accuracy and F1 are therefore easier to interpret here than they would be on a heavily imbalanced dataset. The separate test split also gives the project a clean evaluation set rather than reporting training performance.


## 2. Compare the baseline with the fine-tuned transformer


In [1]:
# Compare the simple baseline and the final model using the same reported metrics.
results = [
    ("TF-IDF + Logistic Regression", 89.46, 89.46),
    ("DistilBERT", 91.19, 91.19),
]
for model, accuracy, f1 in results:
    print(f"{model:30s} accuracy={accuracy:.2f}%  f1={f1:.2f}%")
print(f"Absolute accuracy gain: {91.19 - 89.46:.2f} percentage points")


TF-IDF + Logistic Regression     accuracy=89.46%  f1=89.46%
DistilBERT                     accuracy=91.19%  f1=91.19%
Absolute accuracy gain: 1.73 percentage points


### What this tells us

DistilBERT improves on the TF-IDF/logistic-regression baseline, but the gain is **1.73 percentage points**, not a dramatic leap. That is still useful evidence: the transformer captures contextual information that the bag-of-words baseline misses, while the baseline remains surprisingly competitive and should be retained as a reference point.


## 3. Read the validation trend rather than only the final score


In [1]:
# The validation sequence shows whether training was still improving by the final epoch.
validation_accuracy = [89.8, 91.0, 91.2]
for epoch, score in enumerate(validation_accuracy, start=1):
    print(f"epoch {epoch}: {score:.1f}%")


epoch 1: 89.8%
epoch 2: 91.0%
epoch 3: 91.2%


### What this tells us

Most of the gain arrives by the second epoch, with only a small improvement in epoch three. That pattern suggests the model was approaching a plateau rather than benefiting from indefinitely longer training. It supports the choice to stop after three epochs instead of treating more training as automatically better.


## 4. Connect model design to the hardware constraint


In [1]:
# These choices were practical engineering decisions, not arbitrary hyperparameters.
print("Base model: distilbert-base-uncased")
print("Max sequence length: 256 tokens")
print("Reviews fitting within 256 tokens: 86.8%")
print("Training hardware: Apple M3 Mac, CPU")
print("Training time: ~3.6 hours")


Base model: distilbert-base-uncased
Max sequence length: 256 tokens
Reviews fitting within 256 tokens: 86.8%
Training hardware: Apple M3 Mac, CPU
Training time: ~3.6 hours


### What this tells us

The model design reflects a real resource trade-off. Limiting sequences to 256 tokens reduced memory demand while still covering most reviews, and DistilBERT made fine-tuning feasible on the available hardware. This is a useful engineering lesson: the best production choice is often the model that meets the accuracy target **within the actual compute budget**.


## 5. Failure analysis matters

The repository records failure modes involving sarcasm, double negatives, short emphatic reviews, and domain shift. These are not edge notes to hide—they define where the model should be trusted less.

### Final interpretation

The project demonstrates a solid end-to-end NLP workflow: exploratory analysis, a classical baseline, transformer fine-tuning, held-out evaluation, deployment, and failure analysis. The 91.19% test accuracy is meaningful for IMDb sentiment classification, but it should not be generalized to other domains without new evaluation.
